<a href="https://colab.research.google.com/github/Abdelali-Benajaji/360-Degree-Image-Viewer/blob/main/TP_Gestion_Donn%C3%A9es_Massive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

Étape 1 — Collecte des données

In [3]:
df = pd.read_csv("/content/sample_data/births.csv")

1.2 Explorer rapidement les données

a) Afficher 5 lignes aléatoires

In [25]:
df.sample(5)

,year,month,day,gender,births,date,jour_semaine,jour_ferie
2154,1971,10,24,F,4090,1971-10-24,Sunday,oui
1987,1971,8,4,M,5398,1971-08-04,Wednesday,non
14478,1988,3,16,F,5439,1988-03-16,Wednesday,non
7338,1978,8,10,M,5159,1978-08-10,Thursday,non
13203,1986,6,21,F,4270,1986-06-21,Saturday,oui


b) Afficher les premières lignes

In [5]:
df.head()

,year,month,day,gender,births
0,1969,1,1.0,F,4046
1,1969,1,1.0,M,4440
2,1969,1,2.0,F,4454
3,1969,1,2.0,M,4548
4,1969,1,3.0,F,4548


c) Dimensions du dataset (nombre_lignes, nombre_colonnes)

In [6]:
df.shape

(15547, 5)

d) Liste des colonnes

In [7]:
df.columns

Index(['year', 'month', 'day', 'gender', 'births'], dtype='object')

e) Informations générales

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15547 entries, 0 to 15546
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    15547 non-null  int64  
 1   month   15547 non-null  int64  
 2   day     15067 non-null  float64
 3   gender  15547 non-null  object 
 4   births  15547 non-null  int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 607.4+ KB


Étape 2 — Détection des problèmes

2.1 Vérifier le type de la colonne day

In [9]:
df["day"].dtype

dtype('float64')

2.2 Conversion de day en entier

Supprimer les lignes contenant des valeurs manquantes

In [11]:
df = df.dropna(subset=["day"])

Conversion de day en entier

In [12]:
df["day"] = df["day"].astype(int)

2.3 Détecter les valeurs aberrantes (jours > 31)

In [13]:
df[df["day"] > 31]

,year,month,day,gender,births
62,1969,1,99,F,26
63,1969,1,99,M,38
126,1969,2,99,F,42
127,1969,2,99,M,48
190,1969,3,99,F,64
...,...,...,...,...,...
14572,1988,4,99,F,1
14635,1988,5,99,F,1
14696,1988,6,99,F,1
14697,1988,6,99,M,1


Étape 3 — Nettoyage des lignes invalides

3.1 Supprimer les jours > 31

In [14]:
df = df[df["day"] <= 31]

3.2 Supprimer les valeurs manquantes

In [15]:
df = df.dropna()

 Étape 4 — Question ciblée du TP

4.1 Afficher les naissances du 25/01/2000

In [19]:

df_25_01_2000 = df[
    (df["year"] == 2000) &
    (df["month"] == 1) &
    (df["day"] == 25)
]

df_25_01_2000


,year,month,day,gender,births,date


Étape 5 — Création d’une vraie date

In [17]:

df["date"] = pd.to_datetime(
    df[["year", "month", "day"]],
    errors="coerce"
)


5.2 Afficher les dates

In [20]:
df["date"].head()

,date
0,1969-01-01
1,1969-01-01
2,1969-01-02
3,1969-01-02
4,1969-01-03


5.3 Supprimer les dates non converties

In [21]:
df = df.dropna(subset=["date"])

Étape 6 : Ajouter la colonne jour férié

3.1 Extraire le jour de la semaine

In [22]:
df["jour_semaine"] = df["date"].dt.day_name()

In [24]:
df["jour_ferie"] = df["jour_semaine"].isin(["Saturday", "Sunday"])
df["jour_ferie"] = df["jour_ferie"].map({True: "oui", False: "non"})

In [26]:
df.sample(5)

,year,month,day,gender,births,date,jour_semaine,jour_ferie
13649,1987,1,28,F,5314,1987-01-28,Wednesday,non
8231,1979,10,18,F,4791,1979-10-18,Thursday,non
9145,1981,1,7,F,5001,1981-01-07,Wednesday,non
9508,1981,7,4,F,4373,1981-07-04,Saturday,oui
227,1969,4,18,M,5040,1969-04-18,Friday,non


Étape 7 :Gestion des valeurs **manquantes**


7.1 Remplacement des valeurs manquantes de births (KNNImputer)

Lien direct avec le cours : Hot Deck / KNN

In [27]:
from sklearn.impute import KNNImputer

In [28]:
imputer_knn = KNNImputer(n_neighbors=2)
df["births"] = imputer_knn.fit_transform(df[["births"]])

7.2 Remplacer les valeurs manquantes de month par le mode

In [29]:
df["month"] = df["month"].fillna(df["month"].mode()[0])

7.3 Remplacer les valeurs manquantes de year par la médiane

In [30]:
df["year"] = df["year"].fillna(df["year"].median())

Conclusion intermédiaire

Après le nettoyage, le jeu de données ne contient plus de valeurs manquantes ni de dates invalides.
Les valeurs aberrantes (jours > 31) ont été supprimées.
Les données manquantes ont été imputées à l’aide de méthodes statistiques adaptées :
médiane pour year, mode pour month et KNNImputer pour births.
Ces traitements améliorent la qualité et la cohérence du dataset pour l’analyse.

Étape 8: Enrichissement : décennie, jour, analyses

Ajouter la colonne décennie

In [31]:
df["decennie"] = (df["year"] // 10) * 10

In [32]:
df.sample(5)

,year,month,day,gender,births,date,jour_semaine,jour_ferie,decennie
13836,1987,4,30,M,5618.0,1987-04-30,Thursday,non,1980
910,1970,3,8,F,4342.0,1970-03-08,Sunday,oui,1970
13049,1986,4,5,F,4374.0,1986-04-05,Saturday,oui,1980
5488,1976,3,4,M,4415.0,1976-03-04,Thursday,non,1970
11298,1983,11,24,M,4194.0,1983-11-24,Thursday,non,1980


Afficher les naissances de décembre 1995

In [34]:
df[(df["year"] == 1995) & (df["month"] == 12)]

,year,month,day,gender,births,date,jour_semaine,jour_ferie,decennie
